In [1]:
import os, os.path as osp
import torch
import numpy as np
import pandas as pd

In [2]:
# sklearn for visualization & metrics
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

In [3]:
from src.utils.args import get_args               
from src.utils.consts import SBERT_MODELS
from src.data.data import get_data
from src.data.split import get_split
from src.embeddings.structural import train_node2vec
from sentence_transformers import SentenceTransformer

/home/ldn24004/miniconda3-10/lib/python3.10/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /home/ldn24004/miniconda3-10/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/home/ldn24004/miniconda3-10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
import sys
nb_dir = os.path.dirname(os.path.abspath("__file__"))
if nb_dir not in sys.path:
    sys.path.append(nb_dir)

In [5]:
nb_dir

'/home/ldn24004/autoprompt'

In [6]:
sys.path

['/home/ldn24004/miniconda3-10/lib/python310.zip',
 '/home/ldn24004/miniconda3-10/lib/python3.10',
 '/home/ldn24004/miniconda3-10/lib/python3.10/lib-dynload',
 '',
 '/home/ldn24004/miniconda3-10/lib/python3.10/site-packages',
 '/home/ldn24004/autoprompt']

In [8]:
from src.model.prompt_buffer import PromptBuffer


In [12]:
# Alternative minimal params (if you prefer not to use get_args)
params = {
    "data_path": "datasets",         # folder containing your stored tensors/CSVs (see data.py)
    "data": "cora",                  # or "pubmed"/"usa"/"europe"/"brazil"
    "emb_model": "minilm", # key in SBERT_MODELS
    "node_text": "title_abstract",   # whatever your config expects for cora/pubmed
    "desc_mode": "orig_only",             # "orig_only" | "llm_only" | "both" (see data.py)
    "topo_csv": "/home/ldn24004/autoprompt/datasets/response/text_rich/cora/topo_features.csv",
    "desc_csv": "/home/ldn24004/autoprompt/datasets/response/text_rich/cora/answer_gemma3-12b_TANS.csv",
    # any node2vec hyperparams your train_node2vec() expects, e.g.:
    "struct_emb_dim": 128,
    "walk_length": 20,
    "context_size": 10,
    "walks_per_node": 10,
    "p": 1.0,
    "q": 1.0,
    "epochs": 1,
}
data = get_data(params, params["data"])
from src.data.split import get_split
split = get_split(data.y, params)
struct_embs = train_node2vec(data, params)
train_mask = split["train"].bool()
train_idx = torch.arange(data.num_nodes)[train_mask]
X_text   = data.x[train_idx].detach().cpu().float().numpy()
X_struct = struct_embs[train_idx].detach().cpu().float().numpy()
y_train  = data.y[train_idx].detach().cpu().numpy().astype(int)


title_abstract
Desc mode: orig_only
Example node text: 
  Stochastic pro-positionalization of non-determinate background knowledge. It is a well-known fact that propositional learning algorithms require "good" features to perform well in practice. So a major step in data engineering for inductive learning is the construction of good features by domain experts. These features often represent properties of structured objects, where a property typically is the occurrence of a certain substructure having certain properties. To partly automate the process of "feature engineering", we devised an algorithm that searches for features which are defined by such substructures. The algorithm stochastically conducts a top-down search for first-order clauses, where each clause represents a binary feature. It differs from existing algorithms in that its search is not class-blind, and that it is capable of considering clauses ("context") of almost arbitrary length (size). Preliminary experiments are f

KeyboardInterrupt: 

In [ ]:
# Cell 2 — kNN label agreement
def knn_label_agreement(X, labels, k=5, metric="cosine"):
    # Fit a kNN index; we query k+1 to exclude self
    nn = NearestNeighbors(n_neighbors=k+1, metric=metric)
    nn.fit(X)
    dists, idxs = nn.kneighbors(X, return_distance=True)
    # drop self at position 0
    idxs = idxs[:, 1:]
    nbr_labels = labels[idxs]
    same = (nbr_labels == labels[:, None]).astype(np.float32)
    per_node = same.mean(axis=1)
    overall = float(per_node.mean())
    by_class = {}
    for c in np.unique(labels):
        mask = (labels == c)
        by_class[int(c)] = float(per_node[mask].mean())
    return overall, per_node, by_class

k = 5
overall_text, pernode_text, byclass_text = knn_label_agreement(X_text, y_train, k=k, metric="cosine")
overall_struct, pernode_struct, byclass_struct = knn_label_agreement(X_struct, y_train, k=k, metric="cosine")

print(f"[kNN label agreement@{k}] TEXT:     {overall_text:.3f}")
print(f"[kNN label agreement@{k}] STRUCT:   {overall_struct:.3f}")
print("\nPer-class (TEXT):", byclass_text)
print("Per-class (STRUCT):", byclass_struct)
